Builds and ranks Supplier × Category savings opportunities by combining spend, pricing anomalies, maverick spend, supplier risk, and concentration signals to estimate Potential Annual Savings and Negotiation Priority.

**Imports and engine configuration**

In [0]:
# ============================================================
# DB_06_Build_Savings_Opportunity_Engine
#
# Purpose:
# - Combine 2026 procurement spend with:
#       DB_03 supplier-risk predictions
#       DB_05 pricing-anomaly predictions
#       Gold contract / maverick-spend signals
# - Estimate annualized savings potential
# - Calculate negotiation priority
# - Rank Supplier × Category opportunities
# - Persist explainable opportunity outputs to OneLake
#
# Grain:
# SupplierID × CategoryID × PredictionDate
# ============================================================

from datetime import date, datetime, timezone

import calendar
import math

import mlflow

from pyspark.sql import functions as F
from pyspark.sql import Window


# ------------------------------------------------------------
# Prediction / source dates
# ------------------------------------------------------------

AS_OF_DATE = date(
    2026,
    7,
    31
)

PREDICTION_DATE = AS_OF_DATE

SCORING_YEAR = 2026


# ------------------------------------------------------------
# Annualization
# ------------------------------------------------------------

YEAR_START_DATE = date(
    SCORING_YEAR,
    1,
    1
)

YTD_DAY_COUNT = (
    AS_OF_DATE
    -
    YEAR_START_DATE
).days + 1


YEAR_DAY_COUNT = (
    366
    if calendar.isleap(
        SCORING_YEAR
    )
    else 365
)


ANNUALIZATION_FACTOR = (
    YEAR_DAY_COUNT
    /
    YTD_DAY_COUNT
)


# ------------------------------------------------------------
# Savings assumptions
#
# Pricing:
# Positive observed benchmark variance only.
#
# We cap the benchmark variance at 50% because synthetic
# pricing can contain very large extreme z-scores / ratios.
#
# Maverick:
# A conservative 3% recovery assumption is applied to
# maverick spend NOT already assigned a pricing opportunity.
# ------------------------------------------------------------

MAX_PRICING_VARIANCE_PCT = 50.0

MAVERICK_RECOVERY_RATE = 0.03


# ------------------------------------------------------------
# Actionability
# ------------------------------------------------------------

MIN_ACTIONABLE_SAVINGS_EUR = 5000.0


# ------------------------------------------------------------
# Priority-score weights
#
# Must sum to 1.00
# ------------------------------------------------------------

WEIGHT_SAVINGS_POTENTIAL = 0.45

WEIGHT_SPEND_SCALE = 0.20

WEIGHT_PRICING_SIGNAL = 0.15

WEIGHT_MAVERICK_SIGNAL = 0.10

WEIGHT_SUPPLIER_RISK = 0.05

WEIGHT_CONCENTRATION = 0.05


PRIORITY_WEIGHT_SUM = (
    WEIGHT_SAVINGS_POTENTIAL
    +
    WEIGHT_SPEND_SCALE
    +
    WEIGHT_PRICING_SIGNAL
    +
    WEIGHT_MAVERICK_SIGNAL
    +
    WEIGHT_SUPPLIER_RISK
    +
    WEIGHT_CONCENTRATION
)


if not math.isclose(
    PRIORITY_WEIGHT_SUM,
    1.0,
    abs_tol=1e-9
):

    raise ValueError(
        "Negotiation priority weights "
        "must sum to 1.00."
    )


# ------------------------------------------------------------
# Engine metadata
# ------------------------------------------------------------

ENGINE_NAME = (
    "SavingsOpportunityEngine"
)

ENGINE_VERSION = (
    "1.0"
)

ENGINE_STATUS = (
    "Experimental"
)


print(
    "DB_06 configuration loaded."
)

print(
    "As-of date:",
    AS_OF_DATE
)

print(
    "YTD days:",
    YTD_DAY_COUNT
)

print(
    "Annualization factor:",
    round(
        ANNUALIZATION_FACTOR,
        4
    )
)

print(
    "Pricing variance cap:",
    f"{MAX_PRICING_VARIANCE_PCT:.2f}%"
)

print(
    "Maverick recovery rate:",
    f"{MAVERICK_RECOVERY_RATE:.2%}"
)

print(
    "Priority-weight sum:",
    PRIORITY_WEIGHT_SUM
)

DB_06 configuration loaded.
As-of date: 2026-07-31
YTD days: 212
Annualization factor: 1.7217
Pricing variance cap: 50.00%
Maverick recovery rate: 3.00%
Priority-weight sum: 1.0


**Load OneLake Credentials**

In [0]:
# ============================================================
# Load Fabric OneLake credentials securely
# ============================================================

tenant_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-tenant-id"
)

client_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-id"
)

client_secret = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-secret"
)


print(
    "Fabric OneLake credentials loaded securely."
)

Fabric OneLake credentials loaded securely.


**Configure OneLake OAuth**

In [0]:
# ============================================================
# Configure Fabric OneLake OAuth
# ============================================================

spark.conf.set(
    "fs.azure.account.auth.type",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id",
    client_id
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret",
    client_secret
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint",
    (
        f"https://login.microsoftonline.com/"
        f"{tenant_id}/oauth2/token"
    )
)


print(
    "OneLake OAuth configuration applied."
)

OneLake OAuth configuration applied.


**Define input and output paths**

In [0]:
# ============================================================
# DB_06 input and output paths
# ============================================================

GOLD_LAKEHOUSE_ROOT = (
    "<ABFSS PATH>"
    "<LAKEHOUSE ID>"
)


# ------------------------------------------------------------
# Gold input
# ------------------------------------------------------------

FACT_PURCHASE_ORDER_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Tables/fact_purchase_order"
)


# ------------------------------------------------------------
# DB_03 Supplier Risk input
# ------------------------------------------------------------

SUPPLIER_RISK_PREDICTIONS_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Files/ml/supplier_risk/"
    f"predictions_2026"
)


# ------------------------------------------------------------
# DB_05 Pricing Anomaly input
# ------------------------------------------------------------

PRICING_ANOMALY_PREDICTIONS_PATH = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Files/ml/pricing_anomaly/"
    f"scoring_predictions"
)


# ------------------------------------------------------------
# DB_06 working/output area
# ------------------------------------------------------------

SAVINGS_OPPORTUNITY_ML_ROOT = (
    f"{GOLD_LAKEHOUSE_ROOT}/"
    f"Files/ml/savings_opportunity"
)


SAVINGS_OPPORTUNITIES_PATH = (
    f"{SAVINGS_OPPORTUNITY_ML_ROOT}/"
    f"opportunities_2026"
)


SAVINGS_EVIDENCE_PATH = (
    f"{SAVINGS_OPPORTUNITY_ML_ROOT}/"
    f"opportunity_evidence_2026"
)


ENGINE_METADATA_PATH = (
    f"{SAVINGS_OPPORTUNITY_ML_ROOT}/"
    f"engine_metadata"
)


OPPORTUNITY_SUMMARY_PATH = (
    f"{SAVINGS_OPPORTUNITY_ML_ROOT}/"
    f"opportunity_summary"
)


print(
    "Gold PO:",
    FACT_PURCHASE_ORDER_PATH
)

print(
    "Supplier risk:",
    SUPPLIER_RISK_PREDICTIONS_PATH
)

print(
    "Pricing anomaly:",
    PRICING_ANOMALY_PREDICTIONS_PATH
)

print(
    "Savings opportunities:",
    SAVINGS_OPPORTUNITIES_PATH
)

**Read DB_06 source datasets**

In [0]:
# ============================================================
# Read DB_06 source datasets
# ============================================================

fact_po_df = (
    spark.read
    .format("delta")
    .load(
        FACT_PURCHASE_ORDER_PATH
    )
)


supplier_risk_df = (
    spark.read
    .format("delta")
    .load(
        SUPPLIER_RISK_PREDICTIONS_PATH
    )
)


pricing_anomaly_df = (
    spark.read
    .format("delta")
    .load(
        PRICING_ANOMALY_PREDICTIONS_PATH
    )
)


fact_po_count = (
    fact_po_df.count()
)


supplier_risk_count = (
    supplier_risk_df.count()
)


pricing_anomaly_count = (
    pricing_anomaly_df.count()
)


print(
    "fact_purchase_order:",
    f"{fact_po_count:,}"
)

print(
    "Supplier-risk predictions:",
    f"{supplier_risk_count:,}"
)

print(
    "Pricing-anomaly predictions:",
    f"{pricing_anomaly_count:,}"
)

fact_purchase_order: 75,994
Supplier-risk predictions: 356
Pricing-anomaly predictions: 21,752


**Validate source feature contracts**

In [0]:
# ============================================================
# Validate DB_06 source contracts
# ============================================================

required_po_columns = [
    "POItemID",
    "POID",
    "SupplierID",
    "CategoryID",
    "MaterialID",
    "ContractID",
    "OrderDate",
    "EligibleSpendEUR",
    "LineAmountEUR",
    "SpendEligibilityFlag",
    "MaverickSpendFlag",
    "ContractComplianceFlag"
]


required_pricing_columns = [
    "POItemID",
    "SupplierName",
    "CategoryName",

    "PricingAnomalyScore",
    "PricingAnomalyFlag",

    "PriceComplianceExceptionFlag",

    "ContractPriceVariancePct",

    "HasContractBenchmarkFlag",

    "HasMaterialHistoryFlag",
    "HasSupplierMaterialHistoryFlag",

    "PriceVsMaterialHistoricalAvgPct",
    "PriceVsSupplierMaterialHistoricalAvgPct",

    "PriceVsCategoryHistoricalAvgPct",

    "BenchmarkCoverageCount",

    "ModelName",
    "ModelRunID"
]


required_supplier_risk_columns = [
    "SupplierID",
    "SupplierRiskScore",
    "PredictedHighRiskFlag",
    "ModelName",
    "ModelRunID"
]


missing_po_columns = [
    column_name
    for column_name in required_po_columns
    if column_name
    not in fact_po_df.columns
]


missing_pricing_columns = [
    column_name
    for column_name in required_pricing_columns
    if column_name
    not in pricing_anomaly_df.columns
]


missing_risk_columns = [
    column_name
    for column_name in required_supplier_risk_columns
    if column_name
    not in supplier_risk_df.columns
]


if missing_po_columns:

    raise ValueError(
        "fact_purchase_order is missing: "
        +
        ", ".join(
            missing_po_columns
        )
    )


if missing_pricing_columns:

    raise ValueError(
        "DB_05 pricing predictions are missing: "
        +
        ", ".join(
            missing_pricing_columns
        )
    )


if missing_risk_columns:

    raise ValueError(
        "DB_03 supplier-risk predictions are missing: "
        +
        ", ".join(
            missing_risk_columns
        )
    )


print(
    "DB_06 source contracts PASSED."
)

DB_06 source contracts PASSED.


**Build eligible 2026 PO population**

Use the same governed spend basis as the procurement KPIs.

In [0]:
# ============================================================
# Build eligible 2026 procurement population
# ============================================================

po_2026_df = (
    fact_po_df

    # --------------------------------------------------------
    # Current scoring year
    # --------------------------------------------------------

    .filter(
        F.year(
            F.col(
                "OrderDate"
            )
        )
        ==
        F.lit(
            SCORING_YEAR
        )
    )

    # --------------------------------------------------------
    # Only transactions available as of prediction date
    # --------------------------------------------------------

    .filter(
        F.to_date(
            F.col(
                "OrderDate"
            )
        )
        <=
        F.lit(
            AS_OF_DATE
        )
    )

    # --------------------------------------------------------
    # Eligible procurement spend only
    #
    # SpendEligibilityFlag is BOOLEAN in Gold.
    # --------------------------------------------------------

    .filter(
        F.col(
            "SpendEligibilityFlag"
        )
        ==
        F.lit(
            True
        )
    )

    # --------------------------------------------------------
    # Canonical DB_06 PO-item population
    # --------------------------------------------------------

    .select(
        F.col(
            "POItemID"
        )
        .cast(
            "string"
        )
        .alias(
            "POItemID"
        ),

        F.col(
            "POID"
        )
        .cast(
            "string"
        )
        .alias(
            "POID"
        ),

        F.col(
            "SupplierID"
        )
        .cast(
            "string"
        )
        .alias(
            "SupplierID"
        ),

        F.col(
            "CategoryID"
        )
        .cast(
            "string"
        )
        .alias(
            "CategoryID"
        ),

        F.col(
            "MaterialID"
        )
        .cast(
            "string"
        )
        .alias(
            "MaterialID"
        ),

        F.col(
            "ContractID"
        )
        .cast(
            "string"
        )
        .alias(
            "ContractID"
        ),

        F.to_date(
            F.col(
                "OrderDate"
            )
        )
        .alias(
            "OrderDate"
        ),

        F.coalesce(
            F.col(
                "EligibleSpendEUR"
            )
            .cast(
                "double"
            ),
            F.lit(
                0.0
            )
        )
        .alias(
            "EligibleSpendEUR"
        ),

        F.coalesce(
            F.col(
                "LineAmountEUR"
            )
            .cast(
                "double"
            ),
            F.lit(
                0.0
            )
        )
        .alias(
            "LineAmountEUR"
        ),

        # ----------------------------------------------------
        # Normalize Boolean Gold flags to 1 / 0 for DB_06
        # ----------------------------------------------------

        F.when(
            F.col(
                "MaverickSpendFlag"
            )
            ==
            F.lit(
                True
            ),
            F.lit(
                1
            )
        )
        .otherwise(
            F.lit(
                0
            )
        )
        .cast(
            "int"
        )
        .alias(
            "MaverickSpendFlag"
        ),

        F.when(
            F.col(
                "ContractComplianceFlag"
            )
            ==
            F.lit(
                True
            ),
            F.lit(
                1
            )
        )
        .otherwise(
            F.lit(
                0
            )
        )
        .cast(
            "int"
        )
        .alias(
            "ContractComplianceFlag"
        )
    )
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

po_2026_count = (
    po_2026_df.count()
)


po_2026_spend_eur = (
    po_2026_df

    .agg(
        F.sum(
            "EligibleSpendEUR"
        )
        .alias(
            "EligibleSpendEUR"
        )
    )

    .first()[
        "EligibleSpendEUR"
    ]
)


po_2026_spend_eur = (
    float(
        po_2026_spend_eur
    )
    if po_2026_spend_eur is not None
    else 0.0
)


print(
    "Eligible 2026 PO items:",
    f"{po_2026_count:,}"
)

print(
    "2026 YTD eligible spend EUR:",
    f"{po_2026_spend_eur:,.2f}"
)


# ------------------------------------------------------------
# Confirm normalized schema
# ------------------------------------------------------------

print(
    "\nDB_06 PO population schema:"
)

po_2026_df.printSchema()

Eligible 2026 PO items: 20,632
2026 YTD eligible spend EUR: 2,217,536,128.00

DB_06 PO population schema:
root
 |-- POItemID: string (nullable = true)
 |-- POID: string (nullable = true)
 |-- SupplierID: string (nullable = true)
 |-- CategoryID: string (nullable = true)
 |-- MaterialID: string (nullable = true)
 |-- ContractID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- EligibleSpendEUR: double (nullable = false)
 |-- LineAmountEUR: double (nullable = false)
 |-- MaverickSpendFlag: integer (nullable = false)
 |-- ContractComplianceFlag: integer (nullable = false)



**Prepare DB_05 pricing signals**

In [0]:
# ============================================================
# Prepare DB_05 pricing signals
# ============================================================

pricing_signal_df = (
    pricing_anomaly_df

    .select(
        "POItemID",

        "SupplierName",
        "CategoryName",

        F.col(
            "PricingAnomalyScore"
        )
        .cast("double")
        .alias(
            "PricingAnomalyScore"
        ),

        F.col(
            "PricingAnomalyFlag"
        )
        .cast("int")
        .alias(
            "PricingAnomalyFlag"
        ),

        F.coalesce(
            F.col(
                "PriceComplianceExceptionFlag"
            )
            .cast("int"),
            F.lit(0)
        )
        .alias(
            "PriceComplianceExceptionFlag"
        ),

        F.col(
            "ContractPriceVariancePct"
        )
        .cast("double")
        .alias(
            "ContractPriceVariancePct"
        ),

        F.coalesce(
            F.col(
                "HasContractBenchmarkFlag"
            )
            .cast("int"),
            F.lit(0)
        )
        .alias(
            "HasContractBenchmarkFlag"
        ),

        F.coalesce(
            F.col(
                "HasMaterialHistoryFlag"
            )
            .cast("int"),
            F.lit(0)
        )
        .alias(
            "HasMaterialHistoryFlag"
        ),

        F.coalesce(
            F.col(
                "HasSupplierMaterialHistoryFlag"
            )
            .cast("int"),
            F.lit(0)
        )
        .alias(
            "HasSupplierMaterialHistoryFlag"
        ),

        F.col(
            "PriceVsMaterialHistoricalAvgPct"
        )
        .cast("double")
        .alias(
            "PriceVsMaterialHistoricalAvgPct"
        ),

        F.col(
            "PriceVsSupplierMaterialHistoricalAvgPct"
        )
        .cast("double")
        .alias(
            "PriceVsSupplierMaterialHistoricalAvgPct"
        ),

        F.col(
            "PriceVsCategoryHistoricalAvgPct"
        )
        .cast("double")
        .alias(
            "PriceVsCategoryHistoricalAvgPct"
        ),

        F.col(
            "BenchmarkCoverageCount"
        )
        .cast("int")
        .alias(
            "BenchmarkCoverageCount"
        ),

        F.col(
            "ModelName"
        )
        .alias(
            "PricingModelName"
        ),

        F.col(
            "ModelRunID"
        )
        .alias(
            "PricingModelRunID"
        )
    )
)


pricing_signal_duplicate_count = (
    pricing_signal_df

    .groupBy(
        "POItemID"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


if pricing_signal_duplicate_count > 0:

    raise ValueError(
        "DB_05 pricing output contains "
        "duplicate POItemID values."
    )


print(
    "Pricing signals prepared."
)

Pricing signals prepared.


**Join Gold spend with pricing-anomaly evidence**

In [0]:
# ============================================================
# Join eligible procurement spend to DB_05 pricing signals
# ============================================================

item_opportunity_df = (
    po_2026_df.alias("po")

    .join(
        pricing_signal_df.alias("pa"),

        on="POItemID",

        how="left"
    )

    .withColumn(
        "HasPricingSignalFlag",

        F.when(
            F.col(
                "PricingAnomalyScore"
            ).isNotNull(),
            1
        )

        .otherwise(
            0
        )
    )

    .withColumn(
        "PricingAnomalyFlag",

        F.coalesce(
            F.col(
                "PricingAnomalyFlag"
            ),
            F.lit(0)
        )
    )

    .withColumn(
        "PriceComplianceExceptionFlag",

        F.coalesce(
            F.col(
                "PriceComplianceExceptionFlag"
            ),
            F.lit(0)
        )
    )
)


pricing_signal_covered_count = (
    item_opportunity_df

    .filter(
        F.col(
            "HasPricingSignalFlag"
        )
        == 1
    )

    .count()
)


pricing_signal_coverage_pct = (
    pricing_signal_covered_count
    /
    po_2026_count
    *
    100.0
    if po_2026_count > 0
    else 0.0
)


print(
    "PO items with DB_05 pricing signal:",
    f"{pricing_signal_covered_count:,}"
)

print(
    "Pricing-signal coverage:",
    f"{pricing_signal_coverage_pct:.2f}%"
)

PO items with DB_05 pricing signal: 20,632
Pricing-signal coverage: 100.00%


**Select the savings benchmark per PO item**

The benchmark precedence:

1. Negotiated contract benchmark
2. Same supplier + same material history
3. Same material history across suppliers

Category-average variance are not used directly to calculate euro savings, because different materials within the same category may have fundamentally different unit prices. Category deviation remains useful as an anomaly signal, but not as a direct cash benchmark.

In [0]:
# ============================================================
# Select conservative positive pricing benchmark
# ============================================================

item_opportunity_df = (
    item_opportunity_df

    .withColumn(
        "PricingSavingsBenchmarkType",

        F.when(
            (
                F.col(
                    "HasContractBenchmarkFlag"
                )
                == 1
            )
            &
            (
                F.col(
                    "ContractPriceVariancePct"
                )
                > 0
            )
            &
            (
                (
                    F.col(
                        "PricingAnomalyFlag"
                    )
                    == 1
                )
                |
                (
                    F.col(
                        "PriceComplianceExceptionFlag"
                    )
                    == 1
                )
            ),

            F.lit(
                "CONTRACT"
            )
        )

        .when(
            (
                F.col(
                    "PricingAnomalyFlag"
                )
                == 1
            )
            &
            (
                F.col(
                    "HasSupplierMaterialHistoryFlag"
                )
                == 1
            )
            &
            (
                F.col(
                    "PriceVsSupplierMaterialHistoricalAvgPct"
                )
                > 0
            ),

            F.lit(
                "SUPPLIER_MATERIAL_HISTORY"
            )
        )

        .when(
            (
                F.col(
                    "PricingAnomalyFlag"
                )
                == 1
            )
            &
            (
                F.col(
                    "HasMaterialHistoryFlag"
                )
                == 1
            )
            &
            (
                F.col(
                    "PriceVsMaterialHistoricalAvgPct"
                )
                > 0
            ),

            F.lit(
                "MATERIAL_HISTORY"
            )
        )

        .otherwise(
            F.lit(None)
            .cast("string")
        )
    )


    # --------------------------------------------------------
    # Raw positive benchmark variance
    # --------------------------------------------------------

    .withColumn(
        "SelectedPositivePriceVariancePct",

        F.when(
            F.col(
                "PricingSavingsBenchmarkType"
            )
            ==
            "CONTRACT",

            F.col(
                "ContractPriceVariancePct"
            )
        )

        .when(
            F.col(
                "PricingSavingsBenchmarkType"
            )
            ==
            "SUPPLIER_MATERIAL_HISTORY",

            F.col(
                "PriceVsSupplierMaterialHistoricalAvgPct"
            )
        )

        .when(
            F.col(
                "PricingSavingsBenchmarkType"
            )
            ==
            "MATERIAL_HISTORY",

            F.col(
                "PriceVsMaterialHistoricalAvgPct"
            )
        )

        .otherwise(
            F.lit(None)
            .cast("double")
        )
    )


    # --------------------------------------------------------
    # Conservative cap
    # --------------------------------------------------------

    .withColumn(
        "CappedPositivePriceVariancePct",

        F.when(
            F.col(
                "SelectedPositivePriceVariancePct"
            ).isNotNull(),

            F.least(
                F.col(
                    "SelectedPositivePriceVariancePct"
                ),

                F.lit(
                    MAX_PRICING_VARIANCE_PCT
                )
            )
        )
    )
)


print(
    "Pricing savings benchmarks selected."
)

Pricing savings benchmarks selected.


**Calculate PO-item-level savings evidence**

The pricing formula converts variance relative to benchmark into the estimated excess embedded in current spend:

- Actual price = Benchmark × (1 + variance)
- Excess = Actual Spend × variance / (1 + variance)

In [0]:
# ============================================================
# Calculate PO-item-level savings opportunity
# ============================================================

item_opportunity_df = (
    item_opportunity_df

    # --------------------------------------------------------
    # Pricing opportunity
    #
    # pct / (100 + pct) converts benchmark variance into
    # estimated excess embedded in actual spend.
    # --------------------------------------------------------

    .withColumn(
        "PricingOpportunityYTDEUR",

        F.when(
            (
                F.col(
                    "CappedPositivePriceVariancePct"
                ).isNotNull()
            )
            &
            (
                F.col(
                    "CappedPositivePriceVariancePct"
                )
                > 0
            ),

            F.col(
                "EligibleSpendEUR"
            )
            *
            (
                F.col(
                    "CappedPositivePriceVariancePct"
                )
                /
                (
                    F.lit(
                        100.0
                    )
                    +
                    F.col(
                        "CappedPositivePriceVariancePct"
                    )
                )
            )
        )

        .otherwise(
            F.lit(
                0.0
            )
        )
    )


    # --------------------------------------------------------
    # Maverick spend opportunity
    #
    # Do not count a line again if it already carries a
    # pricing recovery opportunity.
    # --------------------------------------------------------

    .withColumn(
        "MaverickNonPricingSpendYTDEUR",

        F.when(
            (
                F.col(
                    "MaverickSpendFlag"
                )
                == 1
            )
            &
            (
                F.col(
                    "PricingOpportunityYTDEUR"
                )
                <= 0
            ),

            F.col(
                "EligibleSpendEUR"
            )
        )

        .otherwise(
            F.lit(
                0.0
            )
        )
    )

    .withColumn(
        "MaverickOpportunityYTDEUR",

        F.col(
            "MaverickNonPricingSpendYTDEUR"
        )
        *
        F.lit(
            MAVERICK_RECOVERY_RATE
        )
    )


    # --------------------------------------------------------
    # Total non-overlapping line opportunity
    # --------------------------------------------------------

    .withColumn(
        "TotalOpportunityYTDEUR",

        F.col(
            "PricingOpportunityYTDEUR"
        )
        +
        F.col(
            "MaverickOpportunityYTDEUR"
        )
    )


    # --------------------------------------------------------
    # Evidence-spend measures
    # --------------------------------------------------------

    .withColumn(
        "PricingAnomalySpendYTDEUR",

        F.when(
            F.col(
                "PricingAnomalyFlag"
            )
            == 1,

            F.col(
                "EligibleSpendEUR"
            )
        )

        .otherwise(
            F.lit(
                0.0
            )
        )
    )

    .withColumn(
        "MaverickSpendYTDEUR",

        F.when(
            F.col(
                "MaverickSpendFlag"
            )
            == 1,

            F.col(
                "EligibleSpendEUR"
            )
        )

        .otherwise(
            F.lit(
                0.0
            )
        )
    )

    .withColumn(
        "ContractPriceExceptionSpendYTDEUR",

        F.when(
            F.col(
                "PriceComplianceExceptionFlag"
            )
            == 1,

            F.col(
                "EligibleSpendEUR"
            )
        )

        .otherwise(
            F.lit(
                0.0
            )
        )
    )
)


print(
    "PO-item savings evidence calculated."
)

PO-item savings evidence calculated.


**Inspect savings benchmark usage**

In [0]:
# ============================================================
# Inspect pricing benchmark usage
# ============================================================

display(
    item_opportunity_df

    .groupBy(
        "PricingSavingsBenchmarkType"
    )

    .agg(
        F.count("*")
        .alias(
            "POItemCount"
        ),

        F.round(
            F.sum(
                "EligibleSpendEUR"
            ),
            2
        )
        .alias(
            "EligibleSpendEUR"
        ),

        F.round(
            F.sum(
                "PricingOpportunityYTDEUR"
            ),
            2
        )
        .alias(
            "PricingOpportunityYTDEUR"
        )
    )

    .orderBy(
        F.desc(
            "PricingOpportunityYTDEUR"
        )
    )
)

PricingSavingsBenchmarkType,POItemCount,EligibleSpendEUR,PricingOpportunityYTDEUR
MATERIAL_HISTORY,381,5.328190131E7,1.763668473E7
CONTRACT,329,6.247767821E7,1.690130824E7
SUPPLIER_MATERIAL_HISTORY,318,2.835361473E7,9447056.56
null,19604,2.07342293375E9,0.0


**Aggregate to Supplier x Category**

In [0]:
# ============================================================
# Aggregate savings evidence to Supplier × Category
# ============================================================

supplier_category_ytd_df = (
    item_opportunity_df

    .groupBy(
        "SupplierID",
        "CategoryID"
    )

    .agg(
        F.max(
            "SupplierName"
        )
        .alias(
            "SupplierName"
        ),

        F.max(
            "CategoryName"
        )
        .alias(
            "CategoryName"
        ),

        F.count(
            "*"
        )
        .cast("long")
        .alias(
            "POItemCount"
        ),

        F.countDistinct(
            "POID"
        )
        .cast("long")
        .alias(
            "POCount"
        ),

        F.countDistinct(
            "MaterialID"
        )
        .cast("long")
        .alias(
            "MaterialCount"
        ),

        F.round(
            F.sum(
                "EligibleSpendEUR"
            ),
            2
        )
        .alias(
            "EligibleSpendYTDEUR"
        ),

        F.round(
            F.sum(
                "PricingAnomalySpendYTDEUR"
            ),
            2
        )
        .alias(
            "PricingAnomalySpendYTDEUR"
        ),

        F.sum(
            "PricingAnomalyFlag"
        )
        .cast("long")
        .alias(
            "PricingAnomalyPOItemCount"
        ),

        F.round(
            F.avg(
                "PricingAnomalyScore"
            ),
            4
        )
        .alias(
            "AveragePricingAnomalyScore"
        ),

        F.round(
            F.max(
                "PricingAnomalyScore"
            ),
            4
        )
        .alias(
            "MaximumPricingAnomalyScore"
        ),

        F.round(
            F.sum(
                "MaverickSpendYTDEUR"
            ),
            2
        )
        .alias(
            "MaverickSpendYTDEUR"
        ),

        F.sum(
            "MaverickSpendFlag"
        )
        .cast("long")
        .alias(
            "MaverickPOItemCount"
        ),

        F.round(
            F.sum(
                "ContractPriceExceptionSpendYTDEUR"
            ),
            2
        )
        .alias(
            "ContractPriceExceptionSpendYTDEUR"
        ),

        F.sum(
            "PriceComplianceExceptionFlag"
        )
        .cast("long")
        .alias(
            "PriceComplianceExceptionCount"
        ),

        F.round(
            F.sum(
                "PricingOpportunityYTDEUR"
            ),
            2
        )
        .alias(
            "PricingOpportunityYTDEUR"
        ),

        F.round(
            F.sum(
                "MaverickOpportunityYTDEUR"
            ),
            2
        )
        .alias(
            "MaverickOpportunityYTDEUR"
        ),

        F.round(
            F.sum(
                "TotalOpportunityYTDEUR"
            ),
            2
        )
        .alias(
            "PotentialSavingsYTDEUR"
        ),

        F.round(
            F.avg(
                "BenchmarkCoverageCount"
            ),
            4
        )
        .alias(
            "AverageBenchmarkCoverageCount"
        ),

        F.max(
            "PricingModelName"
        )
        .alias(
            "PricingModelName"
        ),

        F.max(
            "PricingModelRunID"
        )
        .alias(
            "PricingModelRunID"
        )
    )
)


supplier_category_count = (
    supplier_category_ytd_df.count()
)


print(
    "Supplier-category combinations:",
    f"{supplier_category_count:,}"
)

Supplier-category combinations: 983


**Annualize spend and savings**

In [0]:
# ============================================================
# Annualize YTD procurement opportunity
# ============================================================

supplier_category_df = (
    supplier_category_ytd_df

    .withColumn(
        "PredictionDate",

        F.lit(
            PREDICTION_DATE
        )
        .cast("date")
    )

    .withColumn(
        "AnnualizationFactor",

        F.lit(
            ANNUALIZATION_FACTOR
        )
    )

    .withColumn(
        "AnnualizedEligibleSpendEUR",

        F.round(
            F.col(
                "EligibleSpendYTDEUR"
            )
            *
            F.lit(
                ANNUALIZATION_FACTOR
            ),
            2
        )
    )

    .withColumn(
        "AnnualizedPricingOpportunityEUR",

        F.round(
            F.col(
                "PricingOpportunityYTDEUR"
            )
            *
            F.lit(
                ANNUALIZATION_FACTOR
            ),
            2
        )
    )

    .withColumn(
        "AnnualizedMaverickOpportunityEUR",

        F.round(
            F.col(
                "MaverickOpportunityYTDEUR"
            )
            *
            F.lit(
                ANNUALIZATION_FACTOR
            ),
            2
        )
    )

    .withColumn(
        "PotentialAnnualSavingsEUR",

        F.round(
            (
                F.col(
                    "AnnualizedPricingOpportunityEUR"
                )
                +
                F.col(
                    "AnnualizedMaverickOpportunityEUR"
                )
            ),
            2
        )
    )


    # --------------------------------------------------------
    # Rates
    # --------------------------------------------------------

    .withColumn(
        "PricingAnomalyRatePct",

        F.when(
            F.col(
                "POItemCount"
            )
            > 0,

            F.col(
                "PricingAnomalyPOItemCount"
            )
            /
            F.col(
                "POItemCount"
            )
            *
            100.0
        )
    )

    .withColumn(
        "MaverickSpendPct",

        F.when(
            F.col(
                "EligibleSpendYTDEUR"
            )
            > 0,

            F.col(
                "MaverickSpendYTDEUR"
            )
            /
            F.col(
                "EligibleSpendYTDEUR"
            )
            *
            100.0
        )
    )

    .withColumn(
        "PricingOpportunityPct",

        F.when(
            F.col(
                "AnnualizedEligibleSpendEUR"
            )
            > 0,

            F.col(
                "AnnualizedPricingOpportunityEUR"
            )
            /
            F.col(
                "AnnualizedEligibleSpendEUR"
            )
            *
            100.0
        )
    )

    .withColumn(
        "PotentialSavingsPct",

        F.when(
            F.col(
                "AnnualizedEligibleSpendEUR"
            )
            > 0,

            F.col(
                "PotentialAnnualSavingsEUR"
            )
            /
            F.col(
                "AnnualizedEligibleSpendEUR"
            )
            *
            100.0
        )
    )
)


print(
    "Annualized savings opportunity calculated."
)

Annualized savings opportunity calculated.


**Add category concentration context**

In [0]:
# ============================================================
# Add category spend concentration context
# ============================================================

category_window = (
    Window
    .partitionBy(
        "CategoryID"
    )
)


category_spend_rank_window = (
    Window

    .partitionBy(
        "CategoryID"
    )

    .orderBy(
        F.desc(
            "AnnualizedEligibleSpendEUR"
        ),
        "SupplierID"
    )
)


supplier_category_df = (
    supplier_category_df

    .withColumn(
        "CategoryAnnualizedSpendEUR",

        F.sum(
            "AnnualizedEligibleSpendEUR"
        )
        .over(
            category_window
        )
    )

    .withColumn(
        "CategorySupplierCount",

        F.count(
            "*"
        )
        .over(
            category_window
        )
    )

    .withColumn(
        "SupplierCategorySpendSharePct",

        F.when(
            F.col(
                "CategoryAnnualizedSpendEUR"
            )
            > 0,

            F.col(
                "AnnualizedEligibleSpendEUR"
            )
            /
            F.col(
                "CategoryAnnualizedSpendEUR"
            )
            *
            100.0
        )
    )

    .withColumn(
        "CategorySpendRank",

        F.row_number()
        .over(
            category_spend_rank_window
        )
    )
)


print(
    "Category concentration features created."
)

Category concentration features created.


**Prepare and Join Supplier Risk**

In [0]:
# ============================================================
# Add DB_03 supplier-risk predictions
# ============================================================

supplier_risk_signal_df = (
    supplier_risk_df

    .select(
        "SupplierID",

        F.col(
            "SupplierRiskScore"
        )
        .cast("double")
        .alias(
            "SupplierRiskScore"
        ),

        F.col(
            "PredictedHighRiskFlag"
        )
        .cast("int")
        .alias(
            "PredictedHighRiskFlag"
        ),

        F.col(
            "ModelName"
        )
        .alias(
            "SupplierRiskModelName"
        ),

        F.col(
            "ModelRunID"
        )
        .alias(
            "SupplierRiskModelRunID"
        )
    )
)


supplier_risk_duplicate_count = (
    supplier_risk_signal_df

    .groupBy(
        "SupplierID"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


if supplier_risk_duplicate_count > 0:

    raise ValueError(
        "DB_03 supplier-risk output "
        "contains duplicate SupplierID values."
    )


supplier_category_df = (
    supplier_category_df

    .join(
        supplier_risk_signal_df,

        on="SupplierID",

        how="left"
    )

    .withColumn(
        "HasSupplierRiskScoreFlag",

        F.when(
            F.col(
                "SupplierRiskScore"
            ).isNotNull(),
            1
        )

        .otherwise(
            0
        )
    )


    # --------------------------------------------------------
    # Neutral value only for priority calculation if risk
    # scoring is unavailable. The original risk score remains
    # null so missing coverage is still transparent.
    # --------------------------------------------------------

    .withColumn(
        "SupplierRiskScoreForPriority",

        F.coalesce(
            F.col(
                "SupplierRiskScore"
            ),
            F.lit(
                50.0
            )
        )
    )
)


print(
    "Supplier-risk signals joined."
)

Supplier-risk signals joined.


**Inspect Supplier Risk coverage**

In [0]:
# ============================================================
# Inspect supplier-risk coverage
# ============================================================

supplier_risk_covered_count = (
    supplier_category_df

    .filter(
        F.col(
            "HasSupplierRiskScoreFlag"
        )
        == 1
    )

    .count()
)


supplier_risk_coverage_pct = (
    supplier_risk_covered_count
    /
    supplier_category_count
    *
    100.0
    if supplier_category_count > 0
    else 0.0
)


print(
    "Supplier-category rows with risk score:",
    f"{supplier_risk_covered_count:,}"
)

print(
    "Supplier-risk coverage:",
    f"{supplier_risk_coverage_pct:.2f}%"
)

Supplier-category rows with risk score: 983
Supplier-risk coverage: 100.00%


**Create normalized negotiation-priority components**

In [0]:
# ============================================================
# Create normalized negotiation-priority components
# ============================================================

savings_percentile_window = (
    Window.orderBy(
        F.col(
            "PotentialAnnualSavingsEUR"
        )
        .asc()
    )
)


spend_percentile_window = (
    Window.orderBy(
        F.col(
            "AnnualizedEligibleSpendEUR"
        )
        .asc()
    )
)


pricing_percentile_window = (
    Window.orderBy(
        F.col(
            "PricingOpportunityPct"
        )
        .asc_nulls_first()
    )
)


maverick_percentile_window = (
    Window.orderBy(
        F.col(
            "MaverickSpendPct"
        )
        .asc_nulls_first()
    )
)


supplier_category_df = (
    supplier_category_df

    .withColumn(
        "SavingsPotentialScore",

        F.percent_rank()
        .over(
            savings_percentile_window
        )
        *
        100.0
    )

    .withColumn(
        "SpendScaleScore",

        F.percent_rank()
        .over(
            spend_percentile_window
        )
        *
        100.0
    )

    .withColumn(
        "PricingSignalScore",

        F.percent_rank()
        .over(
            pricing_percentile_window
        )
        *
        100.0
    )

    .withColumn(
        "MaverickSignalScore",

        F.percent_rank()
        .over(
            maverick_percentile_window
        )
        *
        100.0
    )

    .withColumn(
        "SupplierRiskPriorityScore",

        F.least(
            F.greatest(
                F.col(
                    "SupplierRiskScoreForPriority"
                ),
                F.lit(
                    0.0
                )
            ),
            F.lit(
                100.0
            )
        )
    )

    .withColumn(
        "ConcentrationPriorityScore",

        F.least(
            F.greatest(
                F.coalesce(
                    F.col(
                        "SupplierCategorySpendSharePct"
                    ),
                    F.lit(
                        0.0
                    )
                ),
                F.lit(
                    0.0
                )
            ),
            F.lit(
                100.0
            )
        )
    )
)


print(
    "Priority components normalized."
)

Priority components normalized.


**Calculate negotiation priority score**

In [0]:
# ============================================================
# Calculate negotiation priority score
# ============================================================

supplier_category_df = (
    supplier_category_df

    .withColumn(
        "NegotiationPriorityScoreRaw",

        (
            F.col(
                "SavingsPotentialScore"
            )
            *
            F.lit(
                WEIGHT_SAVINGS_POTENTIAL
            )
        )
        +
        (
            F.col(
                "SpendScaleScore"
            )
            *
            F.lit(
                WEIGHT_SPEND_SCALE
            )
        )
        +
        (
            F.col(
                "PricingSignalScore"
            )
            *
            F.lit(
                WEIGHT_PRICING_SIGNAL
            )
        )
        +
        (
            F.col(
                "MaverickSignalScore"
            )
            *
            F.lit(
                WEIGHT_MAVERICK_SIGNAL
            )
        )
        +
        (
            F.col(
                "SupplierRiskPriorityScore"
            )
            *
            F.lit(
                WEIGHT_SUPPLIER_RISK
            )
        )
        +
        (
            F.col(
                "ConcentrationPriorityScore"
            )
            *
            F.lit(
                WEIGHT_CONCENTRATION
            )
        )
    )


    # --------------------------------------------------------
    # No financial opportunity = no negotiation priority
    # --------------------------------------------------------

    .withColumn(
        "NegotiationPriorityScore",

        F.when(
            F.col(
                "PotentialAnnualSavingsEUR"
            )
            > 0,

            F.round(
                F.col(
                    "NegotiationPriorityScoreRaw"
                ),
                2
            )
        )

        .otherwise(
            F.lit(
                0.0
            )
        )
    )


    # --------------------------------------------------------
    # Priority band
    # --------------------------------------------------------

    .withColumn(
        "NegotiationPriority",

        F.when(
            F.col(
                "NegotiationPriorityScore"
            )
            >= 75,

            F.lit(
                "CRITICAL"
            )
        )

        .when(
            F.col(
                "NegotiationPriorityScore"
            )
            >= 60,

            F.lit(
                "HIGH"
            )
        )

        .when(
            F.col(
                "NegotiationPriorityScore"
            )
            >= 40,

            F.lit(
                "MEDIUM"
            )
        )

        .when(
            F.col(
                "PotentialAnnualSavingsEUR"
            )
            > 0,

            F.lit(
                "LOW"
            )
        )

        .otherwise(
            F.lit(
                "NONE"
            )
        )
    )
)


print(
    "Negotiation priority calculated."
)

Negotiation priority calculated.


**Determine primary savings driver**

In [0]:
# ============================================================
# Determine primary savings-opportunity driver
# ============================================================

supplier_category_df = (
    supplier_category_df

    .withColumn(
        "PrimaryOpportunityDriver",

        F.when(
            F.col(
                "PotentialAnnualSavingsEUR"
            )
            <= 0,

            F.lit(
                "NONE"
            )
        )

        .when(
            (
                F.col(
                    "AnnualizedPricingOpportunityEUR"
                )
                > 0
            )
            &
            (
                F.col(
                    "AnnualizedMaverickOpportunityEUR"
                )
                > 0
            )
            &
            (
                F.abs(
                    F.col(
                        "AnnualizedPricingOpportunityEUR"
                    )
                    -
                    F.col(
                        "AnnualizedMaverickOpportunityEUR"
                    )
                )
                /
                F.greatest(
                    F.col(
                        "AnnualizedPricingOpportunityEUR"
                    ),
                    F.col(
                        "AnnualizedMaverickOpportunityEUR"
                    )
                )
                <= 0.20
            ),

            F.lit(
                "MIXED"
            )
        )

        .when(
            F.col(
                "AnnualizedPricingOpportunityEUR"
            )
            >=
            F.col(
                "AnnualizedMaverickOpportunityEUR"
            ),

            F.lit(
                "PRICING"
            )
        )

        .otherwise(
            F.lit(
                "MAVERICK"
            )
        )
    )

    .withColumn(
        "ActionableOpportunityFlag",

        F.when(
            (
                F.col(
                    "PotentialAnnualSavingsEUR"
                )
                >=
                MIN_ACTIONABLE_SAVINGS_EUR
            )
            &
            (
                F.col(
                    "NegotiationPriorityScore"
                )
                >= 40
            ),

            1
        )

        .otherwise(
            0
        )
    )
)


print(
    "Opportunity-driver classification created."
)

Opportunity-driver classification created.


**Create savings ranks**

In [0]:
# ============================================================
# Rank savings opportunities
# ============================================================

global_savings_rank_window = (
    Window.orderBy(
        F.desc(
            "PotentialAnnualSavingsEUR"
        ),
        F.desc(
            "NegotiationPriorityScore"
        ),
        "SupplierID",
        "CategoryID"
    )
)


category_savings_rank_window = (
    Window

    .partitionBy(
        "CategoryID"
    )

    .orderBy(
        F.desc(
            "PotentialAnnualSavingsEUR"
        ),
        F.desc(
            "NegotiationPriorityScore"
        ),
        "SupplierID"
    )
)


priority_rank_window = (
    Window.orderBy(
        F.desc(
            "NegotiationPriorityScore"
        ),
        F.desc(
            "PotentialAnnualSavingsEUR"
        ),
        "SupplierID",
        "CategoryID"
    )
)


supplier_category_df = (
    supplier_category_df

    .withColumn(
        "SavingsOpportunityRank",

        F.row_number()
        .over(
            global_savings_rank_window
        )
    )

    .withColumn(
        "CategorySavingsRank",

        F.row_number()
        .over(
            category_savings_rank_window
        )
    )

    .withColumn(
        "NegotiationPriorityRank",

        F.row_number()
        .over(
            priority_rank_window
        )
    )
)


print(
    "Savings opportunity ranks created."
)

Savings opportunity ranks created.


**Inspect priority distribution**

In [0]:
# ============================================================
# Inspect negotiation priority distribution
# ============================================================

display(
    supplier_category_df

    .groupBy(
        "NegotiationPriority"
    )

    .agg(
        F.count("*")
        .alias(
            "OpportunityCount"
        ),

        F.sum(
            "ActionableOpportunityFlag"
        )
        .alias(
            "ActionableOpportunityCount"
        ),

        F.round(
            F.sum(
                "AnnualizedEligibleSpendEUR"
            ),
            2
        )
        .alias(
            "AnnualizedSpendEUR"
        ),

        F.round(
            F.sum(
                "PotentialAnnualSavingsEUR"
            ),
            2
        )
        .alias(
            "PotentialAnnualSavingsEUR"
        ),

        F.round(
            F.avg(
                "NegotiationPriorityScore"
            ),
            2
        )
        .alias(
            "AveragePriorityScore"
        )
    )

    .orderBy(
        F.when(
            F.col(
                "NegotiationPriority"
            )
            == "CRITICAL",
            1
        )
        .when(
            F.col(
                "NegotiationPriority"
            )
            == "HIGH",
            2
        )
        .when(
            F.col(
                "NegotiationPriority"
            )
            == "MEDIUM",
            3
        )
        .when(
            F.col(
                "NegotiationPriority"
            )
            == "LOW",
            4
        )
        .otherwise(
            5
        )
    )
)

NegotiationPriority,OpportunityCount,ActionableOpportunityCount,AnnualizedSpendEUR,PotentialAnnualSavingsEUR,AveragePriorityScore
CRITICAL,92,92,2.38045957038E9,7.969145345E7,78.75
HIGH,206,206,9.7311395646E8,1.447265613E7,66.55
MEDIUM,200,173,3.0578998518E8,2866962.82,50.87
LOW,457,0,8.509952251E7,516396.33,23.55
NONE,28,0,7.346473306E7,0.0,0.0


**Inspect highest savings opportunities**

In [0]:
# ============================================================
# Inspect highest-ranked savings opportunities
# ============================================================

display(
    supplier_category_df

    .select(
        "SavingsOpportunityRank",

        "SupplierID",
        "SupplierName",

        "CategoryID",
        "CategoryName",

        "AnnualizedEligibleSpendEUR",

        "PotentialAnnualSavingsEUR",
        "PotentialSavingsPct",

        "AnnualizedPricingOpportunityEUR",
        "AnnualizedMaverickOpportunityEUR",

        "PrimaryOpportunityDriver",

        "PricingAnomalyRatePct",
        "MaverickSpendPct",

        "SupplierCategorySpendSharePct",

        "SupplierRiskScore",
        "PredictedHighRiskFlag",

        "NegotiationPriorityScore",
        "NegotiationPriority",

        "ActionableOpportunityFlag"
    )

    .orderBy(
        "SavingsOpportunityRank"
    )

    .limit(50)
)

SavingsOpportunityRank,SupplierID,SupplierName,CategoryID,CategoryName,AnnualizedEligibleSpendEUR,PotentialAnnualSavingsEUR,PotentialSavingsPct,AnnualizedPricingOpportunityEUR,AnnualizedMaverickOpportunityEUR,PrimaryOpportunityDriver,PricingAnomalyRatePct,MaverickSpendPct,SupplierCategorySpendSharePct,SupplierRiskScore,PredictedHighRiskFlag,NegotiationPriorityScore,NegotiationPriority,ActionableOpportunityFlag
1,SUP000315,Orion Advanced Services PLC,CAT010,Tooling,6.694086624E7,6930149.86,10.352644429717497,6814110.95,116038.91,PRICING,7.228915662650602,28.85803489076178,7.428109577388782,38.10383286169851,0,83.74,CRITICAL,1
2,SUP000242,BluePeak Dynamic Energy Corp.,CAT017,Energy and Utilities,2.621121603E7,6371084.96,24.30671264052757,6361344.63,9740.33,PRICING,16.901408450704224,1.3451964272347745,35.21184870880576,54.74566807204002,1,83.54,CRITICAL,1
3,SUP000243,Titan Integrated Components Corp.,CAT010,Tooling,9.237249741E7,4632023.58,5.014505085253384,4388087.81,243935.77,PRICING,7.207207207207207,12.780133738646557,10.250136743652355,48.05984388113888,1,81.94,CRITICAL,1
4,SUP000295,Lumina Precision Engineering Group,CAT010,Tooling,3.301016243E7,4309287.94,13.054428160231263,3647234.14,662053.8,PRICING,20.588235294117645,100.0,3.6629807391246922,59.63371370017303,1,85.6,CRITICAL,1
5,SUP000353,Aurora Global Systems Group,CAT009,Industrial Equipment,5.411435763E7,3758809.71,6.946048839201567,2346570.3,1412239.41,PRICING,12.5,100.0,2.859432556706994,50.350813725525825,1,83.72,CRITICAL,1
6,SUP000302,Lumina Advanced Mechanical S.A.,CAT010,Tooling,1.00890014E7,3111837.01,30.84385546819331,3086996.68,24840.33,PRICING,50.0,100.0,1.119528505307087,46.11068725786795,1,84.9,CRITICAL,1
7,SUP000419,Ironwood Advanced Chemicals S.A.,CAT009,Industrial Equipment,1.1388218432E8,2988682.0,2.6243630800073507,865070.13,2123611.87,MAVERICK,3.7037037037037033,64.43700841750454,6.017597542227624,53.65899852748203,1,81.93,CRITICAL,1
8,SUP000404,Frontier Global Automation Group,CAT009,Industrial Equipment,1.5619103462E8,2953563.09,1.8909939979498678,2221467.44,732095.65,PRICING,7.865168539325842,18.05153942454332,8.25322056877897,33.96614422806728,0,79.8,CRITICAL,1
9,SUP000404,Frontier Global Automation Group,CAT010,Tooling,5.622731035E7,2591078.32,4.608220282761578,2504250.16,86828.16,PRICING,3.225806451612903,10.731213425387223,6.239277230506995,33.96614422806728,0,80.23,CRITICAL,1
10,SUP000310,Apex Global Energy Ltd.,CAT010,Tooling,5.656039306E7,1953903.68,3.4545440268197454,1749314.5,204589.18,PRICING,6.0606060606060606,16.220040118897444,6.276237834801285,56.60217700136633,1,81.21,CRITICAL,1


**Validate savings-component reconciliation**

In [0]:
# ============================================================
# Validate savings-component reconciliation
# ============================================================

component_difference_count = (
    supplier_category_df

    .filter(
        F.abs(
            F.col(
                "PotentialAnnualSavingsEUR"
            )
            -
            (
                F.col(
                    "AnnualizedPricingOpportunityEUR"
                )
                +
                F.col(
                    "AnnualizedMaverickOpportunityEUR"
                )
            )
        )
        > 0.02
    )

    .count()
)


negative_savings_count = (
    supplier_category_df

    .filter(
        F.col(
            "PotentialAnnualSavingsEUR"
        )
        < 0
    )

    .count()
)


excessive_savings_count = (
    supplier_category_df

    .filter(
        F.col(
            "PotentialAnnualSavingsEUR"
        )
        >
        F.col(
            "AnnualizedEligibleSpendEUR"
        )
    )

    .count()
)


if component_difference_count > 0:

    raise ValueError(
        "PotentialAnnualSavingsEUR does not "
        "reconcile to its components."
    )


if negative_savings_count > 0:

    raise ValueError(
        "Negative savings opportunities detected."
    )


if excessive_savings_count > 0:

    raise ValueError(
        "Savings opportunity exceeds annualized spend."
    )


print(
    "Savings-component reconciliation PASSED."
)

Savings-component reconciliation PASSED.


**Overall portfolio summary**

In [0]:
# ============================================================
# Portfolio-level savings opportunity summary
# ============================================================

portfolio_summary_df = (
    supplier_category_df

    .agg(
        F.count("*")
        .alias(
            "SupplierCategoryOpportunityCount"
        ),

        F.countDistinct(
            "SupplierID"
        )
        .alias(
            "SupplierCount"
        ),

        F.countDistinct(
            "CategoryID"
        )
        .alias(
            "CategoryCount"
        ),

        F.sum(
            "ActionableOpportunityFlag"
        )
        .alias(
            "ActionableOpportunityCount"
        ),

        F.round(
            F.sum(
                "AnnualizedEligibleSpendEUR"
            ),
            2
        )
        .alias(
            "AnnualizedEligibleSpendEUR"
        ),

        F.round(
            F.sum(
                "AnnualizedPricingOpportunityEUR"
            ),
            2
        )
        .alias(
            "AnnualizedPricingOpportunityEUR"
        ),

        F.round(
            F.sum(
                "AnnualizedMaverickOpportunityEUR"
            ),
            2
        )
        .alias(
            "AnnualizedMaverickOpportunityEUR"
        ),

        F.round(
            F.sum(
                "PotentialAnnualSavingsEUR"
            ),
            2
        )
        .alias(
            "PotentialAnnualSavingsEUR"
        )
    )

    .withColumn(
        "PortfolioPotentialSavingsPct",

        F.when(
            F.col(
                "AnnualizedEligibleSpendEUR"
            )
            > 0,

            F.round(
                F.col(
                    "PotentialAnnualSavingsEUR"
                )
                /
                F.col(
                    "AnnualizedEligibleSpendEUR"
                )
                *
                100.0,
                4
            )
        )
    )
)


display(
    portfolio_summary_df
)

SupplierCategoryOpportunityCount,SupplierCount,CategoryCount,ActionableOpportunityCount,AnnualizedEligibleSpendEUR,AnnualizedPricingOpportunityEUR,AnnualizedMaverickOpportunityEUR,PotentialAnnualSavingsEUR,PortfolioPotentialSavingsPct
983,337,20,471,3.81792776759E9,7.572897681E7,2.181849192E7,9.754746873E7,2.555


**DB_06 quality gates**

In [0]:
# ============================================================
# DB_06 Savings Opportunity Engine quality gates
# ============================================================

opportunity_row_count = (
    supplier_category_df.count()
)


duplicate_grain_count = (
    supplier_category_df

    .groupBy(
        "SupplierID",
        "CategoryID",
        "PredictionDate"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        > 1
    )

    .count()
)


null_key_count = (
    supplier_category_df

    .filter(
        F.col(
            "SupplierID"
        ).isNull()
        |
        F.col(
            "CategoryID"
        ).isNull()
        |
        F.col(
            "PredictionDate"
        ).isNull()
    )

    .count()
)


invalid_priority_score_count = (
    supplier_category_df

    .filter(
        F.col(
            "NegotiationPriorityScore"
        ).isNull()
        |
        (
            F.col(
                "NegotiationPriorityScore"
            )
            < 0
        )
        |
        (
            F.col(
                "NegotiationPriorityScore"
            )
            > 100
        )
    )

    .count()
)


invalid_rank_count = (
    supplier_category_df

    .filter(
        F.col(
            "SavingsOpportunityRank"
        )
        <= 0
    )

    .count()
)


invalid_potential_savings_pct_count = (
    supplier_category_df

    .filter(
        F.col(
            "PotentialSavingsPct"
        )
        > 40.0
    )

    .count()
)


actionable_opportunity_count = (
    supplier_category_df

    .filter(
        F.col(
            "ActionableOpportunityFlag"
        )
        == 1
    )

    .count()
)


positive_opportunity_count = (
    supplier_category_df

    .filter(
        F.col(
            "PotentialAnnualSavingsEUR"
        )
        > 0
    )

    .count()
)


quality_checks = [

    (
        "Savings opportunity output contains rows",
        opportunity_row_count > 0
    ),

    (
        "Supplier-category-prediction date grain is unique",
        duplicate_grain_count == 0
    ),

    (
        "Business keys are complete",
        null_key_count == 0
    ),

    (
        "Savings components reconcile",
        component_difference_count == 0
    ),

    (
        "Potential savings are non-negative",
        negative_savings_count == 0
    ),

    (
        "Potential savings do not exceed spend",
        excessive_savings_count == 0
    ),

    (
        "Potential savings percentage is plausible",
        invalid_potential_savings_pct_count == 0
    ),

    (
        "Negotiation priority score is between 0 and 100",
        invalid_priority_score_count == 0
    ),

    (
        "Savings ranks are valid",
        invalid_rank_count == 0
    ),

    (
        "Portfolio contains positive savings opportunities",
        positive_opportunity_count > 0
    ),

    (
        "Portfolio contains actionable opportunities",
        actionable_opportunity_count > 0
    ),

    (
        "Pricing anomaly coverage exceeds 95%",
        pricing_signal_coverage_pct >= 95.0
    ),

    (
        "Supplier risk coverage exceeds 95%",
        supplier_risk_coverage_pct >= 95.0
    )
]


failed_checks = []


for (
    check_name,
    passed
) in quality_checks:

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{check_name}"
    )


    if not passed:

        failed_checks.append(
            check_name
        )


print(
    "\nDB_06 COVERAGE DIAGNOSTICS"
)

print(
    "Pricing-signal coverage:",
    f"{pricing_signal_coverage_pct:.2f}%"
)

print(
    "Supplier-risk coverage:",
    f"{supplier_risk_coverage_pct:.2f}%"
)

print(
    "Positive opportunities:",
    f"{positive_opportunity_count:,}"
)

print(
    "Actionable opportunities:",
    f"{actionable_opportunity_count:,}"
)


if failed_checks:

    raise ValueError(
        "DB_06 quality gate FAILED: "
        +
        "; ".join(
            failed_checks
        )
    )


print(
    "\nDB_06 SAVINGS OPPORTUNITY QUALITY GATE PASSED."
)

PASS | Savings opportunity output contains rows
PASS | Supplier-category-prediction date grain is unique
PASS | Business keys are complete
PASS | Savings components reconcile
PASS | Potential savings are non-negative
PASS | Potential savings do not exceed spend
PASS | Potential savings percentage is plausible
PASS | Negotiation priority score is between 0 and 100
PASS | Savings ranks are valid
PASS | Portfolio contains positive savings opportunities
PASS | Portfolio contains actionable opportunities
PASS | Pricing anomaly coverage exceeds 95%
PASS | Supplier risk coverage exceeds 95%

DB_06 COVERAGE DIAGNOSTICS
Pricing-signal coverage: 100.00%
Supplier-risk coverage: 100.00%
Positive opportunities: 955
Actionable opportunities: 471

DB_06 SAVINGS OPPORTUNITY QUALITY GATE PASSED.


**Track the decision-engine run in MLflow**

In [0]:
# ============================================================
# Track Savings Opportunity Engine execution in MLflow
# ============================================================

portfolio_metrics = (
    portfolio_summary_df
    .first()
)


if mlflow.active_run() is not None:

    mlflow.end_run()


with mlflow.start_run(
    run_name=(
        "savings_opportunity_engine_"
        "2026"
    )
) as engine_run:

    mlflow.set_tags({
        "project":
            (
                "Enterprise Procurement "
                "Intelligence Platform"
            ),

        "engine_family":
            "Savings Opportunity",

        "engine_name":
            ENGINE_NAME,

        "engine_version":
            ENGINE_VERSION,

        "engine_status":
            ENGINE_STATUS,

        "engine_type":
            "PrescriptiveRulesBased",

        "prediction_date":
            str(
                PREDICTION_DATE
            ),

        "output_grain":
            (
                "SupplierID x CategoryID "
                "x PredictionDate"
            )
    })


    mlflow.log_params({
        "as_of_date":
            str(
                AS_OF_DATE
            ),

        "scoring_year":
            SCORING_YEAR,

        "annualization_factor":
            ANNUALIZATION_FACTOR,

        "pricing_variance_cap_pct":
            MAX_PRICING_VARIANCE_PCT,

        "maverick_recovery_rate":
            MAVERICK_RECOVERY_RATE,

        "minimum_actionable_savings_eur":
            MIN_ACTIONABLE_SAVINGS_EUR,

        "weight_savings_potential":
            WEIGHT_SAVINGS_POTENTIAL,

        "weight_spend_scale":
            WEIGHT_SPEND_SCALE,

        "weight_pricing_signal":
            WEIGHT_PRICING_SIGNAL,

        "weight_maverick_signal":
            WEIGHT_MAVERICK_SIGNAL,

        "weight_supplier_risk":
            WEIGHT_SUPPLIER_RISK,

        "weight_concentration":
            WEIGHT_CONCENTRATION
    })


    mlflow.log_metrics({
        "opportunity_count":
            float(
                opportunity_row_count
            ),

        "positive_opportunity_count":
            float(
                positive_opportunity_count
            ),

        "actionable_opportunity_count":
            float(
                actionable_opportunity_count
            ),

        "annualized_eligible_spend_eur":
            float(
                portfolio_metrics[
                    "AnnualizedEligibleSpendEUR"
                ]
            ),

        "annualized_pricing_opportunity_eur":
            float(
                portfolio_metrics[
                    "AnnualizedPricingOpportunityEUR"
                ]
            ),

        "annualized_maverick_opportunity_eur":
            float(
                portfolio_metrics[
                    "AnnualizedMaverickOpportunityEUR"
                ]
            ),

        "potential_annual_savings_eur":
            float(
                portfolio_metrics[
                    "PotentialAnnualSavingsEUR"
                ]
            ),

        "portfolio_potential_savings_pct":
            float(
                portfolio_metrics[
                    "PortfolioPotentialSavingsPct"
                ]
            ),

        "pricing_signal_coverage_pct":
            float(
                pricing_signal_coverage_pct
            ),

        "supplier_risk_coverage_pct":
            float(
                supplier_risk_coverage_pct
            )
    })


    ENGINE_RUN_ID = (
        engine_run.info.run_id
    )


print(
    "Savings Opportunity Engine run tracked."
)

print(
    "Engine Run ID:",
    ENGINE_RUN_ID
)

Savings Opportunity Engine run tracked.
Engine Run ID: a0879dc3778f41f1963d130b42b2f86b


**Add final lineage metadata**

In [0]:
# ============================================================
# Add DB_06 engine lineage metadata
# ============================================================

savings_opportunities_df = (
    supplier_category_df

    .withColumn(
        "EngineName",

        F.lit(
            ENGINE_NAME
        )
    )

    .withColumn(
        "EngineVersion",

        F.lit(
            ENGINE_VERSION
        )
    )

    .withColumn(
        "EngineStatus",

        F.lit(
            ENGINE_STATUS
        )
    )

    .withColumn(
        "EngineRunID",

        F.lit(
            ENGINE_RUN_ID
        )
    )

    .withColumn(
        "EngineExecutionTimestampUTC",

        F.current_timestamp()
    )

    .drop(
        "NegotiationPriorityScoreRaw",
        "SupplierRiskScoreForPriority"
    )
)


print(
    "Final DB_06 lineage metadata added."
)

print(
    "Final opportunity rows:",
    f"{savings_opportunities_df.count():,}"
)

Final DB_06 lineage metadata added.
Final opportunity rows: 983


**Build engine metadata output**

In [0]:
# ============================================================
# Build Savings Opportunity Engine metadata
# ============================================================

engine_metadata_rows = [
    (
        ENGINE_NAME,
        ENGINE_VERSION,
        ENGINE_STATUS,
        ENGINE_RUN_ID,

        PREDICTION_DATE,

        SCORING_YEAR,

        float(
            ANNUALIZATION_FACTOR
        ),

        float(
            MAX_PRICING_VARIANCE_PCT
        ),

        float(
            MAVERICK_RECOVERY_RATE
        ),

        float(
            MIN_ACTIONABLE_SAVINGS_EUR
        ),

        float(
            WEIGHT_SAVINGS_POTENTIAL
        ),

        float(
            WEIGHT_SPEND_SCALE
        ),

        float(
            WEIGHT_PRICING_SIGNAL
        ),

        float(
            WEIGHT_MAVERICK_SIGNAL
        ),

        float(
            WEIGHT_SUPPLIER_RISK
        ),

        float(
            WEIGHT_CONCENTRATION
        ),

        int(
            opportunity_row_count
        ),

        int(
            positive_opportunity_count
        ),

        int(
            actionable_opportunity_count
        ),

        float(
            pricing_signal_coverage_pct
        ),

        float(
            supplier_risk_coverage_pct
        ),

        float(
            portfolio_metrics[
                "AnnualizedEligibleSpendEUR"
            ]
        ),

        float(
            portfolio_metrics[
                "PotentialAnnualSavingsEUR"
            ]
        ),

        float(
            portfolio_metrics[
                "PortfolioPotentialSavingsPct"
            ]
        )
    )
]


engine_metadata_columns = [
    "EngineName",
    "EngineVersion",
    "EngineStatus",
    "EngineRunID",

    "PredictionDate",

    "ScoringYear",

    "AnnualizationFactor",

    "MaxPricingVariancePct",

    "MaverickRecoveryRate",

    "MinimumActionableSavingsEUR",

    "WeightSavingsPotential",

    "WeightSpendScale",

    "WeightPricingSignal",

    "WeightMaverickSignal",

    "WeightSupplierRisk",

    "WeightConcentration",

    "OpportunityCount",

    "PositiveOpportunityCount",

    "ActionableOpportunityCount",

    "PricingSignalCoveragePct",

    "SupplierRiskCoveragePct",

    "AnnualizedEligibleSpendEUR",

    "PotentialAnnualSavingsEUR",

    "PortfolioPotentialSavingsPct"
]


engine_metadata_df = (
    spark.createDataFrame(
        engine_metadata_rows,
        engine_metadata_columns
    )

    .withColumn(
        "PricingSavingsMethod",

        F.lit(
            (
                "Positive contract / supplier-material / "
                "material benchmark variance; "
                "variance capped at 50%"
            )
        )
    )

    .withColumn(
        "MaverickSavingsMethod",

        F.lit(
            (
                "3% recovery applied only to maverick spend "
                "not already counted as pricing opportunity"
            )
        )
    )

    .withColumn(
        "CreatedTimestampUTC",

        F.current_timestamp()
    )
)


display(
    engine_metadata_df
)

EngineName,EngineVersion,EngineStatus,EngineRunID,PredictionDate,ScoringYear,AnnualizationFactor,MaxPricingVariancePct,MaverickRecoveryRate,MinimumActionableSavingsEUR,WeightSavingsPotential,WeightSpendScale,WeightPricingSignal,WeightMaverickSignal,WeightSupplierRisk,WeightConcentration,OpportunityCount,PositiveOpportunityCount,ActionableOpportunityCount,PricingSignalCoveragePct,SupplierRiskCoveragePct,AnnualizedEligibleSpendEUR,PotentialAnnualSavingsEUR,PortfolioPotentialSavingsPct,PricingSavingsMethod,MaverickSavingsMethod,CreatedTimestampUTC
SavingsOpportunityEngine,1.0,Experimental,a0879dc3778f41f1963d130b42b2f86b,2026-07-31,2026,1.721698113207547,50.0,0.03,5000.0,0.45,0.2,0.15,0.1,0.05,0.05,983,955,471,100.0,100.0,3.81792776759E9,9.754746873E7,2.555,Positive contract / supplier-material / material benchmark variance; variance capped at 50%,3% recovery applied only to maverick spend not already counted as pricing opportunity,2026-08-13T10:08:10.059695Z


**Build priority summary output**

In [0]:
# ============================================================
# Build persisted negotiation-priority summary
# ============================================================

opportunity_summary_df = (
    savings_opportunities_df

    .groupBy(
        "NegotiationPriority"
    )

    .agg(
        F.count("*")
        .cast("long")
        .alias(
            "OpportunityCount"
        ),

        F.sum(
            "ActionableOpportunityFlag"
        )
        .cast("long")
        .alias(
            "ActionableOpportunityCount"
        ),

        F.round(
            F.sum(
                "AnnualizedEligibleSpendEUR"
            ),
            2
        )
        .alias(
            "AnnualizedEligibleSpendEUR"
        ),

        F.round(
            F.sum(
                "PotentialAnnualSavingsEUR"
            ),
            2
        )
        .alias(
            "PotentialAnnualSavingsEUR"
        ),

        F.round(
            F.avg(
                "NegotiationPriorityScore"
            ),
            2
        )
        .alias(
            "AverageNegotiationPriorityScore"
        )
    )

    .withColumn(
        "EngineRunID",

        F.lit(
            ENGINE_RUN_ID
        )
    )

    .withColumn(
        "PredictionDate",

        F.lit(
            PREDICTION_DATE
        )
        .cast("date")
    )

    .withColumn(
        "CreatedTimestampUTC",

        F.current_timestamp()
    )
)


display(
    opportunity_summary_df
)

NegotiationPriority,OpportunityCount,ActionableOpportunityCount,AnnualizedEligibleSpendEUR,PotentialAnnualSavingsEUR,AverageNegotiationPriorityScore,EngineRunID,PredictionDate,CreatedTimestampUTC
NONE,28,0,7.346473306E7,0.0,0.0,a0879dc3778f41f1963d130b42b2f86b,2026-07-31,2026-08-13T10:08:12.253208Z
MEDIUM,200,173,3.0578998518E8,2866962.82,50.87,a0879dc3778f41f1963d130b42b2f86b,2026-07-31,2026-08-13T10:08:12.253208Z
LOW,457,0,8.509952251E7,516396.33,23.55,a0879dc3778f41f1963d130b42b2f86b,2026-07-31,2026-08-13T10:08:12.253208Z
HIGH,206,206,9.7311395646E8,1.447265613E7,66.55,a0879dc3778f41f1963d130b42b2f86b,2026-07-31,2026-08-13T10:08:12.253208Z
CRITICAL,92,92,2.38045957038E9,7.969145345E7,78.75,a0879dc3778f41f1963d130b42b2f86b,2026-07-31,2026-08-13T10:08:12.253208Z


**Persist DB_06 outputs**

In [0]:
# ============================================================
# Persist DB_06 outputs to OneLake
# ============================================================

# ------------------------------------------------------------
# Supplier × Category savings opportunity output
# ------------------------------------------------------------

(
    savings_opportunities_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        SAVINGS_OPPORTUNITIES_PATH
    )
)


# ------------------------------------------------------------
# PO-item evidence for audit / explainability
# ------------------------------------------------------------

(
    item_opportunity_df

    .withColumn(
        "PredictionDate",

        F.lit(
            PREDICTION_DATE
        )
        .cast("date")
    )

    .withColumn(
        "EngineRunID",

        F.lit(
            ENGINE_RUN_ID
        )
    )

    .withColumn(
        "EngineExecutionTimestampUTC",

        F.current_timestamp()
    )

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        SAVINGS_EVIDENCE_PATH
    )
)


# ------------------------------------------------------------
# Engine metadata
# ------------------------------------------------------------

(
    engine_metadata_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        ENGINE_METADATA_PATH
    )
)


# ------------------------------------------------------------
# Priority summary
# ------------------------------------------------------------

(
    opportunity_summary_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .save(
        OPPORTUNITY_SUMMARY_PATH
    )
)


print(
    "DB_06 outputs persisted successfully."
)

DB_06 outputs persisted successfully.


**Persistence validation**

In [0]:
# ============================================================
# Validate DB_06 persisted outputs
# ============================================================

persisted_opportunities_df = (
    spark.read
    .format("delta")
    .load(
        SAVINGS_OPPORTUNITIES_PATH
    )
)


persisted_evidence_df = (
    spark.read
    .format("delta")
    .load(
        SAVINGS_EVIDENCE_PATH
    )
)


persisted_metadata_df = (
    spark.read
    .format("delta")
    .load(
        ENGINE_METADATA_PATH
    )
)


persisted_summary_df = (
    spark.read
    .format("delta")
    .load(
        OPPORTUNITY_SUMMARY_PATH
    )
)


persisted_opportunity_count = (
    persisted_opportunities_df.count()
)


persisted_evidence_count = (
    persisted_evidence_df.count()
)


persisted_metadata_count = (
    persisted_metadata_df.count()
)


persisted_summary_count = (
    persisted_summary_df.count()
)


if (
    persisted_opportunity_count
    !=
    opportunity_row_count
):

    raise ValueError(
        "DB_06 savings-opportunity "
        "persistence count mismatch."
    )


if (
    persisted_evidence_count
    !=
    po_2026_count
):

    raise ValueError(
        "DB_06 evidence persistence "
        "count mismatch."
    )


if persisted_metadata_count != 1:

    raise ValueError(
        "DB_06 engine metadata "
        "persistence validation failed."
    )


if persisted_summary_count <= 0:

    raise ValueError(
        "DB_06 opportunity summary "
        "persistence validation failed."
    )


print(
    "DB_06 persistence validation PASSED."
)

print(
    "Savings opportunities:",
    f"{persisted_opportunity_count:,}"
)

print(
    "PO-item evidence rows:",
    f"{persisted_evidence_count:,}"
)

print(
    "Engine metadata rows:",
    persisted_metadata_count
)

print(
    "Priority summary rows:",
    persisted_summary_count
)

print(
    "\nDB_06 SAVINGS OPPORTUNITY ENGINE PASSED."
)

DB_06 persistence validation PASSED.
Savings opportunities: 983
PO-item evidence rows: 20,632
Engine metadata rows: 1
Priority summary rows: 5

DB_06 SAVINGS OPPORTUNITY ENGINE PASSED.
